# Remove incomplete keypoint samples

This notebook removes every row containing at least one missing or blank value from the normalized train and test CSV files. It then exports the cleaned CSVs and a per-pose sample summary. The source files are not modified.

In [1]:
from pathlib import Path

import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the project directory from the current Jupyter working directory."""
    for directory in (start, *start.parents):
        if (directory / "csv_data" / "normalized_nofilter").is_dir():
            return directory
    raise FileNotFoundError(
        "Could not find csv_data/normalized_nofilter from the current directory."
    )


PROJECT_ROOT = find_project_root()
INPUT_DIR = PROJECT_ROOT / "csv_data" / "normalized_nofilter"
OUTPUT_DIR = PROJECT_ROOT / "csv_data" / "normalized_noempty"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILES = {
    "train": INPUT_DIR / "keypoints_train.csv",
    "test": INPUT_DIR / "keypoints_test.csv",
}

OUTPUT_DIR

WindowsPath('C:/Users/vgohu/Desktop/yoga_project/csv_data/normalized_noempty')

In [2]:
def remove_incomplete_rows(csv_path: Path) -> tuple[pd.DataFrame, int]:
    """Load a CSV and remove rows containing a missing or whitespace-only value."""
    dataframe = pd.read_csv(csv_path)

    # Detect both standard missing values (NaN) and strings containing only spaces.
    blank_cells = dataframe.isna() | dataframe.apply(
        lambda column: column.astype("string").str.strip().eq("")
    )
    incomplete_rows = blank_cells.any(axis=1)
    cleaned = dataframe.loc[~incomplete_rows].copy()

    return cleaned, int(incomplete_rows.sum())

In [3]:
cleaned_datasets = {}
processing_results = []

for split, input_path in CSV_FILES.items():
    cleaned, removed_rows = remove_incomplete_rows(input_path)
    output_path = OUTPUT_DIR / input_path.name
    cleaned.to_csv(output_path, index=False)
    cleaned_datasets[split] = cleaned

    processing_results.append(
        {
            "dataset": split,
            "original_rows": len(cleaned) + removed_rows,
            "removed_rows": removed_rows,
            "clean_rows": len(cleaned),
            "output_file": str(output_path),
        }
    )

processing_summary = pd.DataFrame(processing_results)
processing_summary

,dataset,original_rows,removed_rows,clean_rows,output_file
0,train,1081,39,1042,C:\Users\vgohu\Desktop\yoga_project\csv_data\n...
1,test,470,5,465,C:\Users\vgohu\Desktop\yoga_project\csv_data\n...


In [4]:
train_counts = cleaned_datasets["train"]["label"].value_counts().rename("train_samples")
test_counts = cleaned_datasets["test"]["label"].value_counts().rename("test_samples")

pose_summary = (
    pd.concat([train_counts, test_counts], axis=1)
    .fillna(0)
    .astype(int)
    .rename_axis("pose")
    .sort_index()
)
pose_summary["total_samples"] = (
    pose_summary["train_samples"] + pose_summary["test_samples"]
)
pose_summary = pose_summary.reset_index()

summary_path = OUTPUT_DIR / "pose_sample_summary.csv"
pose_summary.to_csv(summary_path, index=False)
pose_summary

,pose,train_samples,test_samples,total_samples
0,downdog,199,94,293
1,goddess,172,80,252
2,plank,263,115,378
3,tree,159,69,228
4,warrior2,249,107,356


In [5]:
# Confirm that the exports contain no empty values and that counts reconcile.
for split, cleaned in cleaned_datasets.items():
    remaining_blanks = cleaned.isna() | cleaned.apply(
        lambda column: column.astype("string").str.strip().eq("")
    )
    assert not remaining_blanks.any(axis=None), f"Blank value remains in {split} data"

assert pose_summary["train_samples"].sum() == len(cleaned_datasets["train"])
assert pose_summary["test_samples"].sum() == len(cleaned_datasets["test"])

print(f"Cleaned CSVs and summary saved in: {OUTPUT_DIR}")
print("Validation passed: no empty values remain and all pose counts match.")

Cleaned CSVs and summary saved in: C:\Users\vgohu\Desktop\yoga_project\csv_data\normalized_noempty
Validation passed: no empty values remain and all pose counts match.


Cleaned both CSVs and preserved the originals.
- Training: 1,081 → 1,042 rows (39 removed)
- Test: 470 → 465 rows (5 removed)
- Verified all cleaned rows have zero empty fields and retain all 104 columns.

| Pose | Train | Test | Total |
|---|---:|---:|---:|
| downdog | 199 | 94 | 293 |
| goddess | 172 | 80 | 252 |
| plank | 263 | 115 | 378 |
| tree | 159 | 69 | 228 |
| warrior2 | 249 | 107 | 356 |